# R03 · Cookbook Reference Lab: Indexes, QueryEngine and NHySQL

**NHyDB 0.27.1 notebook series · v1 · reference part**

These examples are based on the official
[NHyDB Python Cookbook](https://github.com/nhydb/NHyDB/blob/main/docs/cookbook/PYTHON_COOKBOOK.md) (sections 13–15), which serves as their authoritative reference.
Code and expected output are taken unchanged from the cookbook, and the same examples are run by the
NHyDB test suite.

* Every code cell is an **independent example** – run any cell on its own, or *Run All*.
* Each example creates its own temporary database folder.
* Under each example you find the output printed in the cookbook, to compare with yours.
* The main series (`../main/`) explains the ideas behind these APIs step by step.

**Contents** (9 runnable examples)

* 13. Secondary indexes
* 14. QueryEngine and the Q builder
* 15. NHySQL

In [1]:
import nhydb

print('nhydb', nhydb.ENGINE_VERSION, '· file format', nhydb.FILE_FORMAT_VERSION)
if nhydb.ENGINE_VERSION != '0.27.1':
    print('⚠ These examples were written for NHyDB 0.27.1; results may differ.')

nhydb 0.27.1 · file format 0.16


## 13. Secondary indexes

| Kind | Create | Field types | Answers |
|---|---|---|---|
| VALUE | `create_index(index_id, realm_id, name, field)` | any | equality, ranges |
| TOKEN | `create_token_index(...)` or `create_index(..., kind="TOKEN")` | `utf8` | contains word(s) |
| TRIGRAM | `create_trigram_index(..., ngram=3, casefold=False)` | `utf8` | contains substring |
| FULLTEXT | `create_fulltext_index(...)` | `utf8` | words, phrases, BM25 ranking |

The field can be given by name or by `v`. Indexes are **not** maintained
automatically: call `build_indexes()` after writing. An index that is older
than the data is *stale*; the direct index APIs then raise
`StaleIndexError`, the QueryEngine falls back to scanning, and `verify()`
reports it.

In [2]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "idx.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "staff")
    db.create_shape(1, 1, "person", [
        {"v": 1, "name": "name", "type": "utf8"},
        {"v": 2, "name": "dept", "type": "utf8"},
        {"v": 3, "name": "salary", "type": "u32"},
    ])
    db.create_classic_layout(1, 1, "staff", shape_id=1)
    db.put_dense_batch(1, 1, [["Ann", "IT", 5200], ["Ben", "Sales", 4100],
                              ["Cid", "IT", 6100], ["Dee", "HR", 3900]])
    db.create_index(1, 1, "by_dept", "dept")
    db.create_index(2, 1, "by_salary", "salary")
    report = db.build_indexes(commit=True)
    print([(i["name"], i["kind"], i["distinct_values"]) for i in report["indexes"]])

    print(list(db.find_u(1, {"by_dept": "IT"})))                       # equality -> u
    print(list(db.find_u(1, {"by_salary": (4000, 5500)})))             # range (inclusive)
    print([r["name"] for r in db.find(1, {"by_dept": "IT", "by_salary": (6000, 9999)})])
    print(db.posting_count(1, "IT"))                                   # count without reading records
    print(list(db.index_candidates(2, "between", 4000, 6000)))
    print(db.index_candidate_estimate(1, "eq", "IT"))

    db.put(1, 5, ["Eve", "IT", 4800], commit=True)                     # index is now stale
    try:
        list(db.find_u(1, {"by_dept": "IT"}))
    except nhydb.StaleIndexError:
        print("stale -> rebuild")
    db.build_indexes([1, 2], commit=True)
    print(list(db.find_u(1, {"by_dept": "IT"})))

[('by_dept', 'VALUE', 3), ('by_salary', 'VALUE', 4)]
[1, 3]
[1, 2]
['Cid']
2
[1, 2]
{'estimated_rows': 2, 'exact': True, 'key_count': 1}
stale -> rebuild
[1, 3, 5]


*Expected output (from the cookbook):*

```text
[('by_dept', 'VALUE', 3), ('by_salary', 'VALUE', 4)]
[1, 3]
[1, 2]
['Cid']
2
[1, 2]
{'estimated_rows': 2, 'exact': True, 'key_count': 1}
stale -> rebuild
[1, 3, 5]
```

Text indexes (`index_candidates` operations per index kind: TOKEN
`token`/`tokens_all`/`tokens_any`, TRIGRAM `contains`/`icontains`,
FULLTEXT `fulltext`/`phrase`):

In [3]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "text.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "docs")
    db.create_shape(1, 1, "doc", [{"v": 1, "name": "title", "type": "utf8"},
                                  {"v": 2, "name": "body", "type": "utf8"}])
    db.create_classic_layout(1, 1, "docs", shape_id=1)
    db.put_dense_batch(1, 1, [
        ["Rust storage", "an append only storage engine in Rust"],
        ["Python tips", "NumPy arrays and Python speed"],
        ["Rust and Python", "bindings between Rust and Python via PyO3"],
    ])
    db.create_token_index(1, 1, "tok_title", "title")
    db.create_trigram_index(2, 1, "tri_title", "title", casefold=True)
    db.create_fulltext_index(3, 1, "ft_body", "body")
    db.build_indexes(commit=True)

    print(list(db.index_candidates(1, "token", "rust")))              # case-insensitive words
    print(list(db.index_candidates(1, "tokens_all", "rust python")))
    print(list(db.index_candidates(1, "tokens_any", "storage tips")))
    print(list(db.index_candidates(2, "icontains", "PYTH")))           # substring
    print(list(db.index_candidates(3, "fulltext", "rust python")))     # all terms
    print(list(db.index_candidates(3, "phrase", "storage engine")))    # exact word sequence
    print(db.fulltext_corpus(3))
    print(db.fulltext_term_posting(3, "rust"))
    postings, corpus = db.fulltext_payloads(3, ["rust", "python"])
    print(sorted(postings), corpus.total_docs)
    print({i: d.kind for i, d in db.index_defs.items()}, db.index_defs[2].to_dict()["options"])

[1, 3]
[3]
[1, 2]
[2, 3]
[3]
[1]
FullTextCorpus(total_docs=3, total_tokens=19)
(FullTextPosting(u=1, positions=(6,)), FullTextPosting(u=3, positions=(2,)))
['python', 'rust'] 3
{1: 'TOKEN', 2: 'TRIGRAM', 3: 'FULLTEXT'} {'ngram': 3, 'casefold': True}


*Expected output (from the cookbook):*

```text
[1, 3]
[3]
[1, 2]
[2, 3]
[3]
[1]
FullTextCorpus(total_docs=3, total_tokens=19)
(FullTextPosting(u=1, positions=(6,)), FullTextPosting(u=3, positions=(2,)))
['python', 'rust'] 3
{1: 'TOKEN', 2: 'TRIGRAM', 3: 'FULLTEXT'} {'ngram': 3, 'casefold': True}
```

`fulltext_payloads(index_id, terms)` returns the postings of several terms
plus the corpus statistics in one call. Ranked BM25 search is part of the
QueryEngine (section 14) and NHySQL (section 15).

## 14. QueryEngine and the Q builder

`QueryEngine(db)` evaluates expressions built with `Q`. It uses fresh
indexes where possible and scans otherwise.

| Builder | Meaning |
|---|---|
| `Q.field("x") == 5`, `!=`, `<`, `<=`, `>`, `>=` | comparisons |
| `Q.field("x").between(a, b)` | inclusive range |
| `.contains(s)`, `.icontains(s)`, `.token(w)`, `.tokens_all(s)`, `.tokens_any(s)` | text |
| `.fulltext(s)`, `.phrase(s)`, `Q.fulltext(field, s)`, `Q.phrase(field, s)` | full text |
| `.exists()` | field is present |
| `Q.u`, `Q.v`, `Q.realm`, `Q.lane`, `Q.N`, `Q.record_N` | structural predicates |
| `Q.n(*numbers)` | a set of cell numbers |
| `Q.all()` | everything |
| `&`, `|`, `~` | and, or, not |

Return modes: `record` (default), `fields`, `count`, `n`, `cell`,
`aggregate`; `search()` returns `ranked`.

In [4]:
import os
import tempfile

import nhydb
from nhydb.query import Q, QueryEngine

path = os.path.join(tempfile.mkdtemp(), "q.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "sales")
    db.create_shape(1, 1, "sale", [
        {"v": 1, "name": "id", "type": "varuint"},
        {"v": 2, "name": "region", "type": "utf8"},
        {"v": 3, "name": "amount", "type": "f64"},
    ])
    db.create_classic_layout(1, 1, "sales", shape_id=1)
    regions = ["north", "south", "east"]
    db.put_dense_batch(1, 1, [[u, regions[u % 3], float(u * 10)] for u in range(1, 31)])
    db.create_index(1, 1, "by_region", "region")
    db.build_indexes(commit=True)

    q = QueryEngine(db)
    print(q.execute(Q.field("region") == "north", realm=1, return_mode="count")["count"])
    r = q.execute((Q.field("region") == "south") & (Q.field("amount") >= 200), realm=1,
                  return_mode="fields", select=["id", "amount"], order_by=[("amount", "desc")], limit=3)
    print([(row["id"], row["amount"]) for row in r["rows"]], r["count"])
    r = q.execute(Q.all(), realm=1, aggregate={"n": ("count", "*"), "total": ("sum", "amount")},
                  group_by=["region"])
    print(sorted((row["region"], row["n"], row["total"]) for row in r["rows"]))
    print(q.execute(Q.u.between(5, 7), realm=1, return_mode="n")["refs"])
    rec = q.execute(Q.field("id") == 4, realm=1, limit=1)["rows"][0]
    print(rec["data"])
    plan = q.explain(Q.field("region") == "east", realm=1)["plan"]
    print(plan["access"], plan["index_name"])

10
[(28, 280.0), (25, 250.0), (22, 220.0)] 3
[('east', 10, 1550.0), ('north', 10, 1650.0), ('south', 10, 1450.0)]
['N31', 'N43', 'N57']
{'id': 4, 'region': 'south', 'amount': 40.0}
VALUE_INDEX by_region


*Expected output (from the cookbook):*

```text
10
[(28, 280.0), (25, 250.0), (22, 220.0)] 3
[('east', 10, 1550.0), ('north', 10, 1650.0), ('south', 10, 1450.0)]
['N31', 'N43', 'N57']
{'id': 4, 'region': 'south', 'amount': 40.0}
VALUE_INDEX by_region
```

> **Tip:** In 0.27.1 `order_by` of `QueryEngine.execute` only takes
> effect for fields that are also listed in `select`. Include the sort
> fields in `select` (NHySQL `ORDER BY` works without this restriction).

Aggregate operations are `count`, `sum`, `min`, `max`, `avg`. The same
queries are available directly on the database as `db.query(expr, ...)`
and `db.explain_query(expr, ...)`.

**Full-text search with BM25 ranking** (needs a fresh FULLTEXT index):

In [5]:
import os
import tempfile

import nhydb
from nhydb.query import Q, QueryEngine

path = os.path.join(tempfile.mkdtemp(), "search.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "kb")
    db.create_shape(1, 1, "article", [{"v": 1, "name": "title", "type": "utf8"},
                                      {"v": 2, "name": "body", "type": "utf8"}])
    db.create_classic_layout(1, 1, "articles", shape_id=1)
    db.put_dense_batch(1, 1, [
        ["Backups", "copy all three database files while the database is closed"],
        ["Indexes", "build indexes after writing; stale indexes are detected"],
        ["Files", "the key file must be kept safe; without the key file the database cannot be opened"],
    ])
    db.create_fulltext_index(1, 1, "ft_body", "body")
    db.build_indexes(commit=True)

    hits = QueryEngine(db).search(Q.fulltext("body", "key file"), realm=1, select=["title"], limit=5)
    print([(h["title"], round(h["relevance"], 3)) for h in hits["rows"]])
    print(db.query(Q.field("body").phrase("database files"), realm=1, return_mode="count")["count"])

[('Files', 2.417)]
1


*Expected output (from the cookbook):*

```text
[('Files', 2.417)]
1
```

**Queries as JSON-like specifications** (useful for configuration files or
web APIs): `execute_spec(dict)` and `nhydb.query.expr_from_spec(dict)`.

In [6]:
import os
import tempfile

import nhydb
from nhydb.query import QueryEngine, expr_from_spec

path = os.path.join(tempfile.mkdtemp(), "spec.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "r")
    db.create_shape(1, 1, "s", [{"v": 1, "name": "year", "type": "u16"}])
    db.create_classic_layout(1, 1, "c", shape_id=1)
    db.put_dense_batch(1, 1, [[2022], [2024], [2024]], commit=True)

    q = QueryEngine(db)
    print(q.execute_spec({"where": {"field": "year", "op": "eq", "value": 2024}, "realm": 1, "return": "count"})["count"])
    expr = expr_from_spec({"field": "year", "op": "ge", "value": 2023})
    print(expr, q.execute(expr, realm=1, return_mode="count")["count"])

2
FieldPredicate(name='year', op='ge', value=2023, value2=None, options={}) 2


*Expected output (from the cookbook):*

```text
2
FieldPredicate(name='year', op='ge', value=2023, value2=None, options={}) 2
```

**Math providers.** `Q.math(provider, **params)` produces candidate cells
from a computation instead of stored data. Built in are `formula`,
`n_sequence` and `math_layout_inputs` (section 18); your own providers are
plain Python functions `function(engine, context, params)` that return cell
numbers:

In [7]:
import os
import tempfile

import nhydb
from nhydb.query import Q, QueryEngine

path = os.path.join(tempfile.mkdtemp(), "prov.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "squares")
    for k in range(1, 6):
        db.put_cell(1, k, 1, k * k, type="varuint")
    db.commit()

    def even_rows(engine, context, params):
        """Cells (realm 1, u, v=1) for even u up to params['limit']."""
        return [nhydb.dynadron_n(1, u, 1, 0) for u in range(2, params["limit"] + 1, 2)]

    q = QueryEngine(db)
    q.register_math_provider("even_rows", even_rows, description="even u in realm 1")
    print(sorted(q.math_providers))
    r = q.execute(Q.math("even_rows", limit=5), return_mode="cell")
    print([(row["u"], row["value"]) for row in r["rows"]])
    seq = q.execute(Q.math("n_sequence", start=3, stop=20, step=4), return_mode="cell")
    print([(row["N"], row["value"]) for row in seq["rows"]])     # computed cells, stored or not

['even_rows', 'formula', 'math_layout_inputs', 'n_sequence']
[(2, 4), (4, 16)]
[('N3', 1), ('N7', 4), ('N11', None), ('N15', None), ('N19', None)]


*Expected output (from the cookbook):*

```text
['even_rows', 'formula', 'math_layout_inputs', 'n_sequence']
[(2, 4), (4, 16)]
[('N3', 1), ('N7', 4), ('N11', None), ('N15', None), ('N19', None)]
```

## 15. NHySQL

NHySQL is a read-only `SELECT` dialect over CLASSIC layouts. The source is
the layout name (optionally `realm.layout`). Parameters: `?` (positional,
pass a list) and `:name` (pass a dict).

In [8]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "sql.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "shop")
    db.create_shape(1, 1, "order", [
        {"v": 1, "name": "id", "type": "varuint"},
        {"v": 2, "name": "status", "type": "utf8"},
        {"v": 3, "name": "amount", "type": "f64"},
        {"v": 4, "name": "note", "type": "utf8"},
    ])
    db.create_classic_layout(1, 1, "orders", shape_id=1)
    db.put_dense_batch(1, 1, [
        [1, "paid", 19.9, "express delivery"], [2, "new", 5.5, None],
        [3, "paid", 120.0, "gift wrap"], [4, "shipped", 7.25, "express delivery requested"],
    ], commit=True)

    r = db.sql("SELECT status, COUNT(*) AS n, SUM(amount) AS total FROM orders GROUP BY status ORDER BY n DESC")
    print(r["columns"], r["rows"])
    print(db.sql("SELECT id FROM orders WHERE amount BETWEEN ? AND ? ORDER BY id", [5, 20])["rows"])
    print(db.sql("SELECT id FROM orders WHERE status = :s AND note IS NOT NULL", {"s": "paid"})["rows"])
    print(db.sql("SELECT id FROM orders WHERE note CONTAINS 'express' ORDER BY id DESC LIMIT 1")["rows"])
    print(db.sql("SELECT id FROM shop.orders WHERE @u BETWEEN 2 AND 3")["rows"])   # structural u
    print(db.sql("EXPLAIN SELECT COUNT(*) FROM orders WHERE status = 'paid'")["plan"]["access"])

['status', 'n', 'total'] [{'status': 'paid', 'n': 2, 'total': 139.9}, {'status': 'new', 'n': 1, 'total': 5.5}, {'status': 'shipped', 'n': 1, 'total': 7.25}]
[{'id': 1}, {'id': 2}, {'id': 4}]
[{'id': 1}, {'id': 3}]
[{'id': 4}]
[{'id': 2}, {'id': 3}]
MATERIALIZED_SCAN


*Expected output (from the cookbook):*

```text
['status', 'n', 'total'] [{'status': 'paid', 'n': 2, 'total': 139.9}, {'status': 'new', 'n': 1, 'total': 5.5}, {'status': 'shipped', 'n': 1, 'total': 7.25}]
[{'id': 1}, {'id': 2}, {'id': 4}]
[{'id': 1}, {'id': 3}]
[{'id': 4}]
[{'id': 2}, {'id': 3}]
MATERIALIZED_SCAN
```

Full-text relevance and errors:

In [9]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "sql2.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "kb")
    db.create_shape(1, 1, "doc", [{"v": 1, "name": "id", "type": "varuint"},
                                  {"v": 2, "name": "text", "type": "utf8"}])
    db.create_classic_layout(1, 1, "docs", shape_id=1)
    db.put_dense_batch(1, 1, [[1, "quick brown fox"], [2, "quick quick fox"], [3, "lazy dog"]])
    db.create_fulltext_index(1, 1, "ft_text", "text")
    db.build_indexes(commit=True)

    r = db.sql("SELECT id, relevance FROM docs WHERE FULLTEXT(text, :q) ORDER BY relevance DESC LIMIT 5", {"q": "quick"})
    print([row["id"] for row in r["rows"]])
    for bad in ["SELEC id FROM docs", "SELECT DISTINCT id FROM docs", "DELETE FROM docs"]:
        try:
            db.sql(bad)
        except nhydb.SQLSyntaxError as e:
            print("SQLSyntaxError")
    compiled = db.compile_sql("SELECT id FROM docs WHERE id > ?", [1])
    print(compiled.layout_name, compiled.return_mode, compiled.expr)

[2, 1]
SQLSyntaxError
SQLSyntaxError
SQLSyntaxError
docs fields FieldPredicate(name='id', op='gt', value=1, value2=None, options={})


*Expected output (from the cookbook):*

```text
[2, 1]
SQLSyntaxError
SQLSyntaxError
SQLSyntaxError
docs fields FieldPredicate(name='id', op='gt', value=1, value2=None, options={})
```

Structural references (`@u` or `STRUCT.u`, also `realm`, `v`, `lane`,
`n`, `record_n`, `layout_id`, `layout_kind`, `layout_name`, `object`) are
allowed in `WHERE`, not in the select list. Not supported: `DISTINCT`,
`HAVING`, joins and all writing statements.
The same SQL functions exist on the QueryEngine (`execute_sql`,
`compile_sql`, `execute_sql_numpy`); `nhydb.query.parse_sql(text)` only
parses.

More query helpers: SQL through the QueryEngine, `db.explain_query`, the
structural builder `Q.struct(name)`, the parser and the version constants:

In [10]:
import os
import tempfile

import nhydb
from nhydb.query import Q, QUERY_VERSION, SQL_LANGUAGE_VERSION, QueryEngine, parse_sql

path = os.path.join(tempfile.mkdtemp(), "more_q.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "r")
    db.create_shape(1, 1, "s", [{"v": 1, "name": "x", "type": "u8"}])
    db.create_classic_layout(1, 1, "items", shape_id=1)
    db.put_dense_batch(1, 1, [[1], [2], [3]])
    db.create_index(1, 1, "by_x", "x")
    db.build_indexes(commit=True)

    q = QueryEngine(db)
    print(q.execute_sql("SELECT x FROM items WHERE x >= ?", [2])["rows"])
    print(db.explain_query(Q.field("x") == 1, realm=1)["plan"]["access"])
    print(q.execute(Q.struct("layout_name") == "items", realm=1, return_mode="count")["count"])

print(parse_sql("SELECT x FROM items WHERE x > 1").where)
print(QUERY_VERSION, SQL_LANGUAGE_VERSION)

[{'x': 2}, {'x': 3}]
VALUE_INDEX
3
FieldPredicate(name='x', op='gt', value=1, value2=None, options={})
0.27 NHySQL-0.24


*Expected output (from the cookbook):*

```text
[{'x': 2}, {'x': 3}]
VALUE_INDEX
3
FieldPredicate(name='x', op='gt', value=1, value2=None, options={})
0.27 NHySQL-0.24
```